<a href="https://colab.research.google.com/github/Svein-Tore/colab/blob/main/FOPDT-binder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FloatSlider, Button, VBox, HBox, Output, FileUpload
from IPython.display import display, Markdown, HTML
import io
import base64

# === 1. Oppsett for filopplasting ===
uploader = FileUpload(accept='', multiple=False, description="Last opp fil")
main_output = Output()

def start_analysen(change):
    with main_output:
        main_output.clear_output()
        if not uploader.value:
            return

        # Henter fildata (robust metode for ipywidgets 7/8)
        if isinstance(uploader.value, dict):
            file_item = list(uploader.value.values())[0]
        else:
            file_item = uploader.value[0]

        content = file_item['content']
        df = pd.read_csv(io.BytesIO(content), sep=None, engine='python', decimal=',')

        tid_data = df.iloc[:,0].values
        niva_data = df.iloc[:,1].values

        # === 2. Automatisk estimering ===
        y0_est = niva_data[0]
        A_est = niva_data[-1] - y0_est

        thresh10 = y0_est + 0.10 * A_est
        thresh85 = y0_est + 0.85 * A_est
        thresh63 = y0_est + 0.63 * A_est

        t10 = tid_data[np.where(niva_data > thresh10)][0]
        t85 = tid_data[np.where(niva_data > thresh85)][0]
        t63 = tid_data[np.where(niva_data > thresh63)][0]

        L_est = max(0, float(t10 - 0.05 * (t85 - t10)))
        T_est = max(0.1, float(t63 - L_est))

        # === 3. Plott-funksjon med alle hjelpelinjer ===
        def plot_fopdt(A, T, L, y0):
            y_model = np.where(tid_data < L, y0, y0 + A * (1 - np.exp(-(tid_data - L) / T)))
            fig, ax = plt.subplots(figsize=(9, 5))

            ax.plot(tid_data, niva_data, "b.", markersize=3, alpha=0.3, label="Måledata")
            ax.plot(tid_data, y_model, "r-", linewidth=2, label="FOPDT Modell")

            ax.axhline(y0, color='black', linestyle='--', linewidth=1, alpha=0.4)
            ax.text(tid_data[0], y0, f' y0={y0:.2f}', color='black', va='bottom', fontweight='bold')
            ax.axhline(y0 + A, color='black', linestyle='--', linewidth=1, alpha=0.4)

            t_end = tid_data[-1]
            ax.vlines(t_end, ymin=y0, ymax=y0 + A, color='purple', linestyle='-', linewidth=4, label='Δy (Gain)')
            ax.text(t_end, y0 + A/2, f' Δy={A:.2f}', color='purple', fontweight='bold', va='center', ha='left')

            ax.axvline(L, color='orange', linestyle=':', linewidth=2)
            ax.text(L+110, y0+30, f' L={L:.1f}s', color='orange', fontweight='bold', va='bottom', ha='right')

            y63 = y0 + 0.63 * A
            t63 = L + T
            ax.axhline(y63, color='green', linestyle=':', linewidth=1, alpha=0.5)
            ax.axvline(t63, color='green', linestyle=':', linewidth=1, alpha=0.5)
            ax.plot(t63, y63, 'go', markersize=8)
            ax.text(t63, y63, f' y63={y63:.2f}\n T={T:.1f}s', color='green', fontweight='bold', va='top')

            ax.set_xlabel("Tid [s]"); ax.set_ylabel("Nivå")
            ax.grid(True, which='major', linestyle='-', alpha=0.3)
            ax.legend(loc='lower right', fontsize='small')
            return fig

        # === 4. Widgets og Dashbord ===
        style = {'description_width': 'initial'}
        A_slider = FloatSlider(value=A_est, min=A_est*0.5, max=A_est*1.5, step=0.01, description="Δy", style=style, layout={'width': '280px'})
        T_slider = FloatSlider(value=T_est, min=0.1, max=T_est*3, step=0.1, description="T", style=style, layout={'width': '280px'})
        L_slider = FloatSlider(value=L_est, min=0, max=max(tid_data[-1]/2, L_est*5), step=0.1, description="L", style=style, layout={'width': '280px'})
        y0_slider = FloatSlider(value=y0_est, min=y0_est-5, max=y0_est+5, step=0.01, description="y0", style=style, layout={'width': '280px'})

        save_btn = Button(description="Generer bilde", button_style='success', layout={'width': '280px'})
        plot_out = Output(layout={'width': '600px', 'min_width': '600px'})
        download_out = Output(layout={'width': '280px'})

        def update_plot(change):
            with plot_out:
                plot_out.clear_output(wait=True)
                fig = plot_fopdt(A_slider.value, T_slider.value, L_slider.value, y0_slider.value)
                plt.show()

        def download_png(b):
            with download_out:
                download_out.clear_output()
                fig = plot_fopdt(A_slider.value, T_slider.value, L_slider.value, y0_slider.value)
                buf = io.BytesIO()
                fig.savefig(buf, format='png', dpi=300)
                plt.close(fig)
                buf.seek(0)
                b64 = base64.b64encode(buf.read()).decode()
                payload = f"data:image/png;base64,{b64}"
                html = f'<a download="Modell.png" href="{payload}"><button style="width:100%; padding:10px; background:#28a745; color:white; border:none; border-radius:5px; cursor:pointer;">LAGRE PNG</button></a>'
                display(HTML(html))

        for s in [A_slider, T_slider, L_slider, y0_slider]:
            s.observe(update_plot, "value")
        save_btn.on_click(download_png)

        # --- Samlet pedagogisk tekst (Markdown-kolonne) ---
        pedagogisk_tekst = r"""
#### SIMC Reguleringstabell

| Valg av $\lambda$ | Respons | Observasjon |
| :--- | :--- | :--- |
| $\lambda = T/2$ | Rolig | Robust, lite oversving |
| $\lambda = T/4$ | Standard | God balanse |
| $\lambda = T/6$ | Rask | Aggressiv, oversving |

---
#### Beregn PID-parametre
- $K = \frac{\Delta y}{\Delta u}$
- $K_p = \frac{T}{K \cdot (\lambda + L)}$
- $T_i = \min(T, 4 \cdot (\lambda + L))$

---
#### Fremgangsmåte
1. Velg verdi for $\lambda$.
2. Regn ut $K_p$ og $T_i$.
3. Still inn i LabVIEW.
4. Gjør sprangendring.
        """

        info_boks = widgets.Output(layout={'width': '480px', 'padding': '0 0 0 20px', 'border_left': '1px solid #ddd'})
        with info_boks:
            display(Markdown(pedagogisk_tekst))

        slidere_kolonne = VBox([A_slider, T_slider, L_slider, y0_slider, save_btn, download_out], layout={'width': '300px'})
        dashbord = HBox([plot_out, slidere_kolonne, info_boks], layout={'align_items': 'flex-start'})

        display(Markdown(f"**Autoestimat:** Δy={A_est:.2f}, T={T_est:.2f}, L={L_est:.2f}"))
        display(dashbord)
        update_plot(None)

# === 5. Start ===
uploader.observe(start_analysen, names='value')
display(Markdown("# FOPDT Simulator"))
display(Markdown("### Last opp måledata fra fil for å starte FOPDT simulatoren"), uploader, main_output)
